# FatigueGuard Mining — Ingeniería de características con landmarks

Este notebook transforma UTA-RLDD `len60` en un dataset tabular de ventanas temporales. Se
procesa aproximadamente a 5 FPS con MediaPipe Face Landmarker, se calculan EAR, MAR y pose de
cabeza por frame, y después se agregan ventanas no solapadas de 10 segundos.

No se entrena ningún modelo. La comparación descriptiva entre estados usa solamente Train.

## 1. Configuración, rutas y reproducibilidad

In [1]:
import os
os.environ.setdefault("GLOG_minloglevel", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

TARGET_FPS = 5.0
WINDOW_SECONDS = 10.0
EAR_PERCLOS_THRESHOLD = 0.80
EAR_STRONG_REDUCTION_THRESHOLD = 0.60
MAR_HIGH_THRESHOLD = 1.50
ABRUPT_PITCH_DEGREES = 10.0

start = Path.cwd().resolve()
PROJECT_ROOT = start if (start / "data").exists() else start.parent
DATA_ROOT = PROJECT_ROOT / "data/raw/uta_rldd_cropped/len60"
METRICS_DIR = PROJECT_ROOT / "outputs/metrics"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
MODEL_PATH = PROJECT_ROOT / "models/mediapipe/face_landmarker.task"
SPLIT_PATH = PROJECT_ROOT / "data/uta_rldd_subject_split.csv"
PER_FRAME_PATH = METRICS_DIR / "uta_landmarks_per_frame.csv"
CALIBRATION_PATH = METRICS_DIR / "uta_subject_calibration_baselines.csv"
WINDOWS_PATH = PROCESSED_DIR / "uta_rldd_landmark_windows.csv"
SUMMARY_PATH = METRICS_DIR / "uta_landmark_feature_summary.csv"
for folder in [METRICS_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
for path in [DATA_ROOT, MODEL_PATH, SPLIT_PATH]:
    assert path.exists(), f"Falta el recurso requerido: {path}"

experiment_start = time.perf_counter()
print(f"MediaPipe: {mp.__version__}")
print(f"Dataset: {DATA_ROOT}")
print(f"Frecuencia objetivo: {TARGET_FPS} FPS")

MediaPipe: 1.0.0
Dataset: data\raw\uta_rldd_cropped\len60
Frecuencia objetivo: 5.0 FPS


## 2. Inventario mínimo de videos y split existente

Se excluye el sujeto 42 mediante el split ya guardado. Cada video conserva sujeto, estado,
etiqueta y partición. No se genera un split nuevo.

In [2]:
subject_split = pd.read_csv(SPLIT_PATH, dtype={"subject_id": str})
subject_split["subject_id"] = subject_split.subject_id.str.zfill(2)
assert subject_split.subject_id.is_unique
assert "42" not in set(subject_split.subject_id)
split_map = dict(zip(subject_split.subject_id, subject_split.split))

train_subjects = set(subject_split.loc[subject_split.split == "Train", "subject_id"])
validation_subjects = set(subject_split.loc[subject_split.split == "Validation", "subject_id"])
test_subjects = set(subject_split.loc[subject_split.split == "Test", "subject_id"])
assert train_subjects.isdisjoint(validation_subjects)
assert train_subjects.isdisjoint(test_subjects)
assert validation_subjects.isdisjoint(test_subjects)
assert [len(train_subjects), len(validation_subjects), len(test_subjects)] == [41, 9, 9]

video_rows = []
for subject_id in sorted(split_map):
    for state_code, label in [(0, 0), (10, 1)]:
        for video_path in sorted((DATA_ROOT / subject_id / str(state_code)).glob("*.mp4")):
            capture = cv2.VideoCapture(str(video_path))
            assert capture.isOpened(), f"No abre: {video_path}"
            source_fps = float(capture.get(cv2.CAP_PROP_FPS))
            frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            capture.release()
            sample_step = max(1, int(round(source_fps / TARGET_FPS)))
            expected_samples = int(np.ceil(frame_count / sample_step))
            video_rows.append({
                "filepath": video_path.relative_to(PROJECT_ROOT).as_posix(),
                "video_id": video_path.stem,
                "subject_id": subject_id,
                "state_code": state_code,
                "label": label,
                "split": split_map[subject_id],
                "source_fps": source_fps,
                "frame_count": frame_count,
                "sample_step": sample_step,
                "expected_samples": expected_samples,
            })
video_manifest = pd.DataFrame(video_rows)
assert video_manifest.video_id.is_unique
assert len(video_manifest) == 1135
display(video_manifest.groupby(["split", "label"]).size().unstack(fill_value=0))
print(f"Videos a procesar: {len(video_manifest):,}")
print(f"Frames esperados a {TARGET_FPS:g} FPS: {video_manifest.expected_samples.sum():,}")

label,0,1
split,,
Test,97,81
Train,392,397
Validation,82,86


Videos a procesar: 1,135
Frames esperados a 5 FPS: 340,298


## 3. Definición de EAR, MAR y pose de cabeza

**EAR** usa seis landmarks por ojo: suma dos distancias verticales dividida por dos veces la
distancia horizontal. **MAR** promedia tres aperturas verticales de la boca y las divide por
su anchura horizontal. La pose es una aproximación geométrica obtenida con `solvePnP` usando
nariz, mentón, extremos oculares y comisuras. MediaPipe Tasks no expone una confianza escalar
por frame; por eso `detection_quality` queda `NaN` y `valid_face` registra el éxito.

In [3]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH_VERTICAL_PAIRS = [(13, 14), (82, 87), (312, 317)]
MOUTH_CORNERS = (78, 308)
POSE_INDICES = [1, 152, 33, 263, 61, 291]
MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0), (0.0, -63.6, -12.5),
    (-43.3, 32.7, -26.0), (43.3, 32.7, -26.0),
    (-28.9, -28.9, -24.1), (28.9, -28.9, -24.1),
], dtype=np.float64)

def distance(points, index_a, index_b):
    return float(np.linalg.norm(points[index_a] - points[index_b]))

def eye_aspect_ratio(points, indices):
    p1, p2, p3, p4, p5, p6 = indices
    horizontal = distance(points, p1, p4)
    if horizontal <= 1e-8:
        return np.nan
    return (distance(points, p2, p6) + distance(points, p3, p5)) / (2.0 * horizontal)

def mouth_aspect_ratio(points):
    width = distance(points, *MOUTH_CORNERS)
    if width <= 1e-8:
        return np.nan
    vertical = np.mean([distance(points, a, b) for a, b in MOUTH_VERTICAL_PAIRS])
    return vertical / width

def estimate_head_pose(points, width, height):
    image_points = np.array([
        (points[index, 0] * width, points[index, 1] * height)
        for index in POSE_INDICES
    ], dtype=np.float64)
    focal_length = float(width)
    camera_matrix = np.array([
        [focal_length, 0, width / 2], [0, focal_length, height / 2], [0, 0, 1]
    ], dtype=np.float64)
    success, rotation_vector, _ = cv2.solvePnP(
        MODEL_POINTS, image_points, camera_matrix, np.zeros((4, 1)),
        flags=cv2.SOLVEPNP_ITERATIVE,
    )
    if not success:
        return np.nan, np.nan, np.nan
    rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
    angles = cv2.RQDecomp3x3(rotation_matrix)[0]
    return float(angles[0]), float(angles[1]), float(angles[2])

## 4. Extracción incremental a aproximadamente 5 FPS

El CSV conserva también frames sin detección, con métricas `NaN`, para medir calidad sin
ocultar fallos. Antes de procesar se verifica si el archivo ya contiene exactamente el número
esperado por video. Si está completo no se procesa nuevamente. Si está incompleto se reanudan
solamente los videos faltantes o incompletos.

In [4]:
frame_columns = [
    "subject_id", "video_id", "state_code", "label", "split",
    "timestamp", "frame_index", "ear_left", "ear_right", "ear_mean",
    "mar", "pitch", "yaw", "roll", "detection_quality", "valid_face",
]

existing_frame_data = None
completed_video_ids = set()
if PER_FRAME_PATH.exists():
    existing_frame_data = pd.read_csv(PER_FRAME_PATH, dtype={"subject_id": str})
    if set(frame_columns).issubset(existing_frame_data.columns):
        observed_counts = existing_frame_data.groupby("video_id").size()
        expected_counts = video_manifest.set_index("video_id").expected_samples
        completed_video_ids = {
            video_id for video_id in expected_counts.index
            if observed_counts.get(video_id, 0) == expected_counts[video_id]
        }
extraction_complete = len(completed_video_ids) == len(video_manifest)
extraction_complete_at_start = extraction_complete

if extraction_complete:
    print("El CSV por frame está completo; no se reprocesan videos.")
else:
    # Conserva únicamente videos ya completos; elimina bloques parciales antes de reanudarlos.
    if existing_frame_data is not None and completed_video_ids:
        clean_existing = existing_frame_data[
            existing_frame_data.video_id.isin(completed_video_ids)
        ][frame_columns]
        clean_existing.to_csv(PER_FRAME_PATH, index=False)
        write_header = False
    else:
        if PER_FRAME_PATH.exists():
            PER_FRAME_PATH.unlink()
        write_header = True

    options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    pending = video_manifest[~video_manifest.video_id.isin(completed_video_ids)]
    extraction_start = time.perf_counter()
    for sequence, video in enumerate(pending.itertuples(), start=1):
        capture = cv2.VideoCapture(str(PROJECT_ROOT / video.filepath))
        rows = []
        with vision.FaceLandmarker.create_from_options(options) as detector:
            frame_index = 0
            sample_number = 0
            while True:
                success, frame_bgr = capture.read()
                if not success:
                    break
                if frame_index % video.sample_step == 0:
                    timestamp = frame_index / video.source_fps
                    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                    result = detector.detect_for_video(mp_image, int(round(timestamp * 1000)))
                    row = {
                        "subject_id": video.subject_id, "video_id": video.video_id,
                        "state_code": video.state_code, "label": video.label,
                        "split": video.split, "timestamp": timestamp,
                        "frame_index": frame_index, "ear_left": np.nan,
                        "ear_right": np.nan, "ear_mean": np.nan, "mar": np.nan,
                        "pitch": np.nan, "yaw": np.nan, "roll": np.nan,
                        "detection_quality": np.nan, "valid_face": False,
                    }
                    if result.face_landmarks:
                        landmarks = result.face_landmarks[0]
                        points = np.array([(lm.x, lm.y, lm.z) for lm in landmarks], dtype=np.float64)
                        ear_left = eye_aspect_ratio(points, LEFT_EYE)
                        ear_right = eye_aspect_ratio(points, RIGHT_EYE)
                        pitch, yaw, roll = estimate_head_pose(
                            points, frame_bgr.shape[1], frame_bgr.shape[0]
                        )
                        row.update({
                            "ear_left": ear_left, "ear_right": ear_right,
                            "ear_mean": np.nanmean([ear_left, ear_right]),
                            "mar": mouth_aspect_ratio(points),
                            "pitch": pitch, "yaw": yaw, "roll": roll,
                            "valid_face": True,
                        })
                    rows.append(row)
                    sample_number += 1
                frame_index += 1
        capture.release()
        video_frame_data = pd.DataFrame(rows, columns=frame_columns)
        assert len(video_frame_data) == video.expected_samples
        video_frame_data.to_csv(
            PER_FRAME_PATH, mode="a", header=write_header, index=False
        )
        write_header = False
        if sequence % 50 == 0 or sequence == len(pending):
            elapsed = time.perf_counter() - extraction_start
            rate = sequence / elapsed if elapsed else 0
            remaining_minutes = (len(pending) - sequence) / rate / 60 if rate else np.nan
            print(
                f"Videos {sequence}/{len(pending)} | "
                f"tiempo {elapsed / 60:.1f} min | restante aprox. {remaining_minutes:.1f} min"
            )

landmarks = pd.read_csv(PER_FRAME_PATH, dtype={"subject_id": str})
landmarks["subject_id"] = landmarks.subject_id.str.zfill(2)
observed_counts = landmarks.groupby("video_id").size()
expected_counts = video_manifest.set_index("video_id").expected_samples
assert len(landmarks) == int(expected_counts.sum())
assert all(observed_counts.get(video_id, 0) == count for video_id, count in expected_counts.items())
print(f"Frames analizados: {len(landmarks):,}")
print(f"Detección facial: {landmarks.valid_face.mean():.2%}")

El CSV por frame está completo; no se reprocesan videos.


Frames analizados: 340,298
Detección facial: 99.88%


## 5. Calibración personal inicial por sujeto

Para simular el funcionamiento futuro de FatigueGuard se elige determinísticamente el primer
video `state_code=0` de cada sujeto y se reservan sus primeros 30 segundos como calibración.
No se usa ningún video Drowsy ni estadísticas futuras. Los frames de calibración se excluyen
después de todas las ventanas de Train, Validation y Test.

- `EAR_base`: percentil 90 del EAR válido durante calibración.
- `MAR_base`: mediana del MAR válido durante calibración.
- `pitch_base`, `yaw_base`, `roll_base`: centros circulares robustos durante calibración.
- `EAR_relative = EAR / EAR_base`; `MAR_relative = MAR / MAR_base`.
- Pose relativa: ángulo observado menos su baseline personal.

**PERCLOS relativo** en una ventana es la fracción de frames con EAR válido donde
`ear_relative < 0.80`. No se utiliza un threshold EAR absoluto universal. Esta calibración
personal forma parte prevista del funcionamiento futuro del sistema para cada operador nuevo.

In [5]:
calibration_video_by_subject = (
    video_manifest[video_manifest.state_code == 0]
    .sort_values(["subject_id", "video_id"])
    .groupby("subject_id").first()["video_id"]
)
landmarks["calibration_video_id"] = landmarks.subject_id.map(calibration_video_by_subject)
landmarks["is_calibration"] = (
    (landmarks.video_id == landmarks.calibration_video_id)
    & (landmarks.state_code == 0)
    & (landmarks.timestamp < 30.0)
)
calibration_frames = landmarks[landmarks.is_calibration].copy()
assert calibration_frames.subject_id.nunique() == len(subject_split)
assert (calibration_frames.state_code == 0).all()
assert (calibration_frames.label == 0).all()

def wrap_angle_degrees(values):
    return (np.asarray(values) + 180.0) % 360.0 - 180.0

def robust_circular_center(values):
    valid = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(valid) == 0:
        return np.nan
    radians = np.deg2rad(valid)
    initial_center = np.rad2deg(np.arctan2(np.mean(np.sin(radians)), np.mean(np.cos(radians))))
    centered = wrap_angle_degrees(valid - initial_center)
    return float(wrap_angle_degrees(initial_center + np.median(centered)))

calibration_baselines = calibration_frames.groupby("subject_id").agg(
    calibration_video_id=("video_id", "first"),
    calibration_start=("timestamp", "min"),
    calibration_end=("timestamp", "max"),
    n_calibration_frames=("timestamp", "size"),
    n_valid_face=("valid_face", "sum"),
    ear_base=("ear_mean", lambda values: values.dropna().quantile(0.90)),
    mar_base=("mar", "median"),
    pitch_base=("pitch", robust_circular_center),
    yaw_base=("yaw", robust_circular_center),
    roll_base=("roll", robust_circular_center),
).reset_index()
calibration_baselines["calibration_face_detection_rate"] = (
    calibration_baselines.n_valid_face / calibration_baselines.n_calibration_frames
)
calibration_baselines["split"] = calibration_baselines.subject_id.map(split_map)
baseline_columns = ["ear_base", "mar_base", "pitch_base", "yaw_base", "roll_base"]
assert calibration_baselines[baseline_columns].notna().all().all()
calibration_baselines.to_csv(CALIBRATION_PATH, index=False)

landmarks = landmarks.merge(
    calibration_baselines[["subject_id", *baseline_columns]],
    on="subject_id", how="left", validate="many_to_one",
)
landmarks["ear_relative"] = landmarks.ear_mean / landmarks.ear_base.replace(0, np.nan)
landmarks["mar_relative"] = landmarks.mar / landmarks.mar_base.replace(0, np.nan)
landmarks["pitch_delta"] = wrap_angle_degrees(landmarks.pitch - landmarks.pitch_base)
landmarks["yaw_delta"] = wrap_angle_degrees(landmarks.yaw - landmarks.yaw_base)
landmarks["roll_delta"] = wrap_angle_degrees(landmarks.roll - landmarks.roll_base)
analysis_landmarks = landmarks[~landmarks.is_calibration].copy()
assert not analysis_landmarks.is_calibration.any()
display(calibration_baselines.describe(include="all").round(4))
print(f"Frames reservados para calibración: {len(calibration_frames):,}")

,subject_id,calibration_video_id,calibration_start,calibration_end,n_calibration_frames,n_valid_face,ear_base,mar_base,pitch_base,yaw_base,roll_base,calibration_face_detection_rate,split
count,59,59,59.0,59.0,59.0,59.0000,59.0000,59.0000,59.0000,59.0000,59.0000,59.0000,59
unique,59,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,01,01_0_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train
freq,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41
mean,NaN,NaN,0.0,29.8,150.0,149.8983,0.2605,0.0145,41.2807,-0.5095,-0.3538,0.9993,NaN
std,NaN,NaN,0.0,0.0,0.0,0.7811,0.0320,0.0182,167.3668,7.1117,2.8448,0.0052,NaN
min,NaN,NaN,0.0,29.8,150.0,144.0000,0.1930,0.0020,-179.7497,-20.7489,-5.1368,0.9600,NaN
25%,NaN,NaN,0.0,29.8,150.0,150.0000,0.2386,0.0054,-171.1117,-4.1212,-2.3979,1.0000,NaN
50%,NaN,NaN,0.0,29.8,150.0,150.0000,0.2547,0.0085,161.8855,0.0664,-0.5835,1.0000,NaN
75%,NaN,NaN,0.0,29.8,150.0,150.0000,0.2760,0.0145,171.5477,3.2588,1.3440,1.0000,NaN


Frames reservados para calibración: 8,850


## 6. Ventanas temporales de 10 segundos

Cada video se divide sin solapamiento. Las condiciones continuas se aproximan a 5 FPS. Un
frame inválido rompe una secuencia de cierre para no inventar continuidad durante fallos de
detección. Las ventanas problemáticas se conservan con sus tasas de validez y valores `NaN`.

In [6]:
def longest_true_run(values):
    longest = current = 0
    for value in values:
        current = current + 1 if bool(value) else 0
        longest = max(longest, current)
    return longest

def episode_count(values):
    array = np.asarray(values, dtype=bool)
    if len(array) == 0:
        return 0
    return int(array[0]) + int(np.sum(array[1:] & ~array[:-1]))

def safe_stat(series, statistic):
    valid = series.dropna()
    return statistic(valid) if len(valid) else np.nan

window_rows = []
for (video_id, window_index), group in analysis_landmarks.groupby([
    "video_id", (analysis_landmarks.timestamp // WINDOW_SECONDS).astype(int)
]):
    group = group.sort_values("timestamp")
    first = group.iloc[0]
    ear_valid = group.ear_relative.notna()
    mar_valid = group.mar_relative.notna()
    closed = (group.ear_relative < EAR_PERCLOS_THRESHOLD) & ear_valid
    strong_reduction = (group.ear_relative < EAR_STRONG_REDUCTION_THRESHOLD) & ear_valid
    elevated_mouth = (group.mar_relative > MAR_HIGH_THRESHOLD) & mar_valid
    pitch_differences = group.pitch_delta.diff().abs()
    window_rows.append({
        "subject_id": first.subject_id,
        "video_id": video_id,
        "state_code": int(first.state_code),
        "label": int(first.label),
        "split": first.split,
        "window_start": float(window_index * WINDOW_SECONDS),
        "window_end": float((window_index + 1) * WINDOW_SECONDS),
        "n_sampled_frames": len(group),
        "n_valid_face": int(group.valid_face.sum()),
        "face_detection_rate": float(group.valid_face.mean()),
        "ear_valid_rate": float(ear_valid.mean()),
        "mar_valid_rate": float(mar_valid.mean()),
        "ear_relative_mean": safe_stat(group.ear_relative, np.mean),
        "ear_relative_std": safe_stat(group.ear_relative, lambda x: x.std(ddof=0)),
        "ear_relative_min": safe_stat(group.ear_relative, np.min),
        "ear_relative_p10": safe_stat(group.ear_relative, lambda x: x.quantile(0.10)),
        "strong_ear_reduction_rate": float(strong_reduction[ear_valid].mean()) if ear_valid.any() else np.nan,
        "perclos_relative": float(closed[ear_valid].mean()) if ear_valid.any() else np.nan,
        "max_eye_closure_seconds": longest_true_run(closed.to_numpy()) / TARGET_FPS,
        "eye_closure_episode_count": episode_count(closed.to_numpy()),
        "mar_relative_mean": safe_stat(group.mar_relative, np.mean),
        "mar_relative_std": safe_stat(group.mar_relative, lambda x: x.std(ddof=0)),
        "mar_relative_max": safe_stat(group.mar_relative, np.max),
        "high_mouth_open_rate": float(elevated_mouth[mar_valid].mean()) if mar_valid.any() else np.nan,
        "pitch_delta_mean": safe_stat(group.pitch_delta, np.mean),
        "pitch_delta_std": safe_stat(group.pitch_delta, lambda x: x.std(ddof=0)),
        "pitch_delta_range": safe_stat(group.pitch_delta, lambda x: x.max() - x.min()),
        "yaw_delta_std": safe_stat(group.yaw_delta, lambda x: x.std(ddof=0)),
        "roll_delta_std": safe_stat(group.roll_delta, lambda x: x.std(ddof=0)),
        "max_head_drop_delta": safe_stat(group.pitch_delta, np.max),
        "abrupt_pitch_movement_count": int((pitch_differences > ABRUPT_PITCH_DEGREES).sum()),
    })

windows = pd.DataFrame(window_rows)
assert windows.split.notna().all()
calibration_video_ids = set(calibration_baselines.calibration_video_id)
assert (windows.loc[windows.video_id.isin(calibration_video_ids), "window_start"] >= 30.0).all()
windows.to_csv(WINDOWS_PATH, index=False)
id_columns = ["subject_id", "video_id", "state_code", "label", "split", "window_start", "window_end"]
feature_columns = [column for column in windows.columns if column not in id_columns]
print(f"Ventanas generadas: {len(windows):,}")
print(f"Features: {len(feature_columns)}")
display(windows.groupby(["split", "label"]).size().unstack(fill_value=0))

Ventanas generadas: 6,633
Features: 24


label,0,1
split,,
Test,555,486
Train,2229,2382
Validation,465,516


## 7. Control de calidad mínimo

No se eliminan ventanas. Se reporta la proporción de NaN por variable y se señalan aquellas
con más de 20 % de valores faltantes.

In [7]:
missing_by_feature = windows[feature_columns].isna().mean().sort_values(ascending=False)
too_many_nan = missing_by_feature[missing_by_feature > 0.20]
overall_numeric_missing = windows[feature_columns].isna().mean().mean()
print(f"Sujetos procesados: {windows.subject_id.nunique()}")
print(f"Videos procesados: {windows.video_id.nunique()}")
print(f"Ventanas: {len(windows):,}")
print(f"Rostro detectado por frame: {landmarks.valid_face.mean():.2%}")
print(f"Promedio de valores faltantes en features: {overall_numeric_missing:.2%}")
print("Variables con más de 20% NaN:")
display(too_many_nan.to_frame("missing_rate"))
print("Balance total:")
display(windows.label.value_counts().rename(index={0: "Non Drowsy", 1: "Drowsy"}).to_frame("windows"))

Sujetos procesados: 59
Videos procesados: 1135
Ventanas: 6,633
Rostro detectado por frame: 99.88%
Promedio de valores faltantes en features: 0.00%
Variables con más de 20% NaN:


,missing_rate


Balance total:


,windows
label,
Drowsy,3384
Non Drowsy,3249


## 8. Validación descriptiva rápida de señal — solo Train

Se comparan estadísticas de variables principales sin entrenar modelos. Validation y Test no
participan en selección ni interpretación de features.

In [8]:
signal_features = [
    "perclos_relative", "ear_relative_mean", "max_eye_closure_seconds",
    "mar_relative_mean", "pitch_delta_mean", "pitch_delta_std",
]
train_windows = windows[windows.split == "Train"].copy()
train_descriptive = (
    train_windows.groupby("label")[signal_features]
    .agg(["count", "mean", "median", "std"])
)
display(train_descriptive.round(4))

summary_rows = []
for feature in signal_features:
    for label, label_name in [(0, "Non Drowsy"), (1, "Drowsy")]:
        values = train_windows.loc[train_windows.label == label, feature].dropna()
        for statistic, value in {
            "count": len(values), "mean": values.mean(),
            "median": values.median(), "std": values.std(),
        }.items():
            summary_rows.append({
                "summary_type": "train_class_descriptive", "feature": feature,
                "group": label_name, "statistic": statistic, "value": value,
            })
    means = train_windows.groupby("label")[feature].mean()
    summary_rows.append({
        "summary_type": "train_class_difference", "feature": feature,
        "group": "Drowsy - Non Drowsy", "statistic": "mean_difference",
        "value": means.get(1, np.nan) - means.get(0, np.nan),
    })
for feature, missing_rate in missing_by_feature.items():
    summary_rows.append({
        "summary_type": "quality", "feature": feature, "group": "All",
        "statistic": "missing_rate", "value": missing_rate,
    })
summary_rows.extend([
    {"summary_type": "quality", "feature": "valid_face", "group": "All", "statistic": "detection_rate", "value": landmarks.valid_face.mean()},
    {"summary_type": "count", "feature": "frames", "group": "All", "statistic": "n", "value": len(landmarks)},
    {"summary_type": "count", "feature": "windows", "group": "All", "statistic": "n", "value": len(windows)},
])
feature_summary = pd.DataFrame(summary_rows)
feature_summary.to_csv(SUMMARY_PATH, index=False)

mean_comparison = train_windows.groupby("label")[signal_features].mean().T
mean_comparison.columns = ["Non Drowsy", "Drowsy"]
mean_comparison["Difference Drowsy - Non Drowsy"] = mean_comparison["Drowsy"] - mean_comparison["Non Drowsy"]
display(mean_comparison.round(4))

perclos_relative                        ear_relative_mean          \
                 count    mean median     std             count    mean   
label                                                                     
0                 2229  0.1049   0.04  0.1519              2229  0.9138   
1                 2382  0.3964   0.26  0.3575              2382  0.7996   

                      max_eye_closure_seconds          ... mar_relative_mean  \
       median     std                   count    mean  ...            median   
label                                                  ...                     
0      0.9233  0.0706                    2229  0.5445  ...            1.0418   
1      0.8139  0.1746                    2382  2.9064  ...            1.3375   

              pitch_delta_mean                          pitch_delta_std  \
          std            count    mean  median      std           count   
label                                                                     
0      1.1200             2229 -0.6462 -0.3187   4.9618            2229   
1      4.5445             2382 -1.0605 -0.9924  10.0205            2382   

                               
         mean  median     std  
label                          
0      1.6897  0.9495  2.9092  
1      2.7149  1.1430  5.1719  

[2 rows x 24 columns]

,Non Drowsy,Drowsy,Difference Drowsy - Non Drowsy
perclos_relative,0.1049,0.3964,0.2916
ear_relative_mean,0.9138,0.7996,-0.1142
max_eye_closure_seconds,0.5445,2.9064,2.3619
mar_relative_mean,1.2485,2.6178,1.3693
pitch_delta_mean,-0.6462,-1.0605,-0.4143
pitch_delta_std,1.6897,2.7149,1.0253


## 9. Conclusiones

In [9]:
train_means = train_windows.groupby("label")[signal_features].mean()
differences = train_means.loc[1] - train_means.loc[0]
detection_rate = landmarks.valid_face.mean()
high_missing_features = too_many_nan.index.tolist()

conclusions = [
    f"1. **Detección facial:** MediaPipe detectó rostro en {detection_rate:.1%} de los frames; "
    + ("la cobertura es mayoritaria." if detection_rate >= 0.80 else "la cobertura requiere cautela."),
    f"2. **Ventanas disponibles:** {len(windows):,} ventanas de 10 segundos, correspondientes a {windows.video_id.nunique():,} videos y {windows.subject_id.nunique()} sujetos.",
    f"3. **EAR/PERCLOS:** en Train, Drowsy - Non Drowsy = {differences['perclos_relative']:+.4f} en PERCLOS y {differences['ear_relative_mean']:+.4f} en EAR relativo medio.",
    f"4. **MAR:** la diferencia de MAR relativo medio es {differences['mar_relative_mean']:+.4f}.",
    f"5. **Head pose:** las diferencias son {differences['pitch_delta_mean']:+.3f}° en pitch delta medio y {differences['pitch_delta_std']:+.3f}° en su variabilidad.",
    f"6. **Faltantes:** variables con más de 20% NaN: {high_missing_features if high_missing_features else 'ninguna'}.",
    f"7. **Preparación para ML clásico:** "
    + ("el dataset queda estructurado y con cobertura suficiente; aún deben definirse reglas explícitas de manejo de NaN usando solo Train." if detection_rate >= 0.80 else "la baja cobertura debe resolverse antes del modelado."),
]
display(Markdown("\n\n".join(conclusions)))

1. **Detección facial:** MediaPipe detectó rostro en 99.9% de los frames; la cobertura es mayoritaria.

2. **Ventanas disponibles:** 6,633 ventanas de 10 segundos, correspondientes a 1,135 videos y 59 sujetos.

3. **EAR/PERCLOS:** en Train, Drowsy - Non Drowsy = +0.2916 en PERCLOS y -0.1142 en EAR relativo medio.

4. **MAR:** la diferencia de MAR relativo medio es +1.3693.

5. **Head pose:** las diferencias son -0.414° en pitch delta medio y +1.025° en su variabilidad.

6. **Faltantes:** variables con más de 20% NaN: ninguna.

7. **Preparación para ML clásico:** el dataset queda estructurado y con cobertura suficiente; aún deben definirse reglas explícitas de manejo de NaN usando solo Train.

## 10. Verificación final

In [10]:
current_run_seconds = time.perf_counter() - experiment_start
extraction_file_seconds = max(
    0.0, PER_FRAME_PATH.stat().st_mtime - PER_FRAME_PATH.stat().st_ctime
)
total_seconds = (
    extraction_file_seconds + current_run_seconds
    if extraction_complete_at_start else current_run_seconds
)
generated_files = [PER_FRAME_PATH, CALIBRATION_PATH, WINDOWS_PATH, SUMMARY_PATH]
assert all(path.exists() for path in generated_files)
assert set(windows.subject_id) == set(subject_split.subject_id)
assert train_subjects.isdisjoint(validation_subjects)
assert train_subjects.isdisjoint(test_subjects)
assert validation_subjects.isdisjoint(test_subjects)
assert len(feature_columns) >= 20 and len(feature_columns) <= 30

print(f"Tiempo total: {total_seconds / 60:.2f} min")
print(f"Frames analizados: {len(landmarks):,}")
print(f"Detección facial: {landmarks.valid_face.mean():.2%}")
print(f"Ventanas generadas: {len(windows):,}")
print(f"Número de features: {len(feature_columns)}")
print("Balance:", windows.label.value_counts().sort_index().to_dict())
print("Archivos generados:")
for path in generated_files:
    print("-", path.relative_to(PROJECT_ROOT).as_posix())

Tiempo total: 47.21 min
Frames analizados: 340,298
Detección facial: 99.88%
Ventanas generadas: 6,633
Número de features: 24
Balance: {0: 3249, 1: 3384}
Archivos generados:
- outputs/metrics/uta_landmarks_per_frame.csv
- outputs/metrics/uta_subject_calibration_baselines.csv
- data/processed/uta_rldd_landmark_windows.csv
- outputs/metrics/uta_landmark_feature_summary.csv
